# BitSemCom + GRAND, plan v2: C=32 benchmark vs GRAND-in-the-loop

Runs the experiments of `docs/colab_runbook.md` end to end. Everything persistent lives in Google Drive
(`MyDrive/BitSemCom`): the pretrained model you upload once, the data zips, the training runs and the results.
The code is cloned fresh from GitHub onto the VM's local disk every session, so nothing has to be zipped or
copied by hand, and no PC (yours or the lab's) is part of the path.

**Before the first run**

1. GitHub: *Settings > Developer settings > Fine-grained personal access tokens*: new token, repository access
   `BitSemCom` only, permission *Contents: read and write*.
2. Colab: key icon in the left sidebar, add a secret named `GH_TOKEN` with that token, switch on *Notebook access*.
   Secrets belong to your Google account, so this works from any browser or PC.
3. Drive: create `MyDrive/BitSemCom/Pretrain/` and upload `BitSemCom_C32_stage2_ep480.model` there (Drive website).
4. Runtime: *Runtime > Change runtime type > GPU*. A100 or L4 for the training cells (bf16); T4 works but is slower (fp16).

**Every session**: run cells 1 to 4 (parameters, Drive and code, packages, data), then whichever experiment cells
you need. The training cells are resumable: if the session dies or the time budget is reached, run the same cell again.

## 1. Parameters

In [ ]:
GITHUB_REPO = "SafaaSalman/BitSemCom"
BRANCH = "grand-implementation"
GIT_NAME, GIT_EMAIL = "Safaa Salman", "safaa.salman.ss@gmail.com"   # author of the results commits made from Colab

DRIVE = "/content/drive/MyDrive/BitSemCom"   # persistent: Pretrain/, data/, runs/, results/
REPO = "/content/BitSemCom"                  # fresh clone on the VM's local disk (fast)
RUNS, RESULTS = DRIVE + "/runs", DRIVE + "/results"
FIG = RESULTS + "/figures"

C = 32
INIT = "Pretrain/BitSemCom_C32_stage2_ep480.model"
STEPS, BATCH = 30000, 16
MAX_HOURS = 11                               # save and stop before the Colab session limit; rerun the cell to continue
SNR_MIN, SNR_MAX = 0, 10                     # training Es/N0 range (dB), drawn per batch
CODE, GRAND_BUDGET = "crc:0x9eb2:128:112", 2000
EVAL_SNRS = "0,2,4,6,8,10,12"
EVAL_CONDITIONS = "awgn,awgn_over3,awgn_under3,fade8,fade8_h0.1"

## 2. Google Drive and the code

If the clone fails, clear this cell's output before sharing the notebook: git may echo the URL that carries the token.

In [ ]:
import os, glob
from google.colab import drive, userdata

drive.mount("/content/drive")
for d in ("Pretrain", "data", "runs", "results"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)

try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception as e:
    raise SystemExit("Add the GitHub token as a Colab secret named GH_TOKEN (key icon on the left) "
                     "and switch on notebook access. Error: %s" % e)

if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q -b {BRANCH} https://{TOKEN}@github.com/{GITHUB_REPO} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/{GITHUB_REPO}
else:
    !git -C {REPO} pull -q https://{TOKEN}@github.com/{GITHUB_REPO} {BRANCH}
%cd {REPO}
!git log --oneline -1

## 3. Packages and GPU

In [ ]:
!pip -q install timm lpips open_clip_torch scipy
import os, torch
from IPython.display import Image as IMG, display

assert torch.cuda.is_available(), "select a GPU runtime: Runtime > Change runtime type"
AMP = "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"   # bf16 on A100 / L4 / H100, fp16 on T4
NPROC = os.cpu_count()
print(torch.cuda.get_device_name(0), "| autocast", AMP, "| cpu cores", NPROC, "| torch", torch.__version__)

## 4. Pretrained model and data

Downloads happen once and are kept on Drive (`data/`). DIV2K is unzipped to the local disk every session because
reading 800 large PNGs through the Drive mount is slow. The 24 Kodak 256x256 centre crops are the validation and
evaluation set.

In [ ]:
import os, glob
from PIL import Image

if not glob.glob(f"{DRIVE}/Pretrain/*C{C}_*.model"):
    raise SystemExit(f"upload the C={C} stage-2 model to {DRIVE}/Pretrain/ first (Drive website)")
!cp {DRIVE}/Pretrain/*.model Pretrain/
!ls -la Pretrain

DIV2K_ZIP = f"{DRIVE}/data/DIV2K_train_HR.zip"       # 800 images, 3.5 GB
if not os.path.exists(DIV2K_ZIP):
    !wget -q -O /content/DIV2K_train_HR.zip http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
    !cp /content/DIV2K_train_HR.zip {DIV2K_ZIP}
if not os.path.isdir("datasets/DIV2K_train_HR"):
    !unzip -q {DIV2K_ZIP} -d datasets/
print("DIV2K images:", len(glob.glob("datasets/DIV2K_train_HR/*.png")))

KODAK = f"{DRIVE}/data/kodak_full"
os.makedirs(KODAK, exist_ok=True)
for i in range(1, 25):
    f = f"{KODAK}/kodim{i:02d}.png"
    url = f"https://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png"
    if not os.path.exists(f):
        !wget -q -O {f} {url}
os.makedirs("datasets/kodak_patches256", exist_ok=True)
for f in sorted(glob.glob(f"{KODAK}/*.png")):
    im = Image.open(f)
    w, h = im.size
    l, t = (w - 256) // 2, (h - 256) // 2
    im.crop((l, t, l + 256, t + 256)).save("datasets/kodak_patches256/" + os.path.basename(f))
print("Kodak patches:", len(glob.glob("datasets/kodak_patches256/*.png")))

## 5. LLR shape study (minutes, no training)

What the semantic decoder and GRAND receive under each channel condition: channel LLR densities by true bit,
P(1) at the decoder input, and the same after the coded link + ORBGRAND. This is the first thing to show the supervisor.

In [ ]:
!python experiments/plot_llr.py --C {C} --limit 8 --snrs 0,4,8,12 --conditions awgn,awgn_over3,awgn_under3,fade8,fade8_h0.1,fade1 --budget 100000 --out {FIG}
for name in ("llr_summary", "llr_awgn", "llr_awgn_over3", "llr_fade8_h0.1"):
    display(IMG(f"{FIG}/{name}.png", width=1000))

## 6. Benchmark training (no GRAND)

AdamW, 500 warmup steps then cosine 1e-4 to 1e-6 over the step budget, per-batch SNR drawn uniformly in
[SNR_MIN, SNR_MAX], validation on the Kodak patches every 1000 steps, best and last checkpoints, early stop after 8
validations without improvement. Expected 0.15 to 0.25 s/step on an A100. Resumable: rerun the cell.

In [ ]:
OUT = f"{RUNS}/C{C}_bench"
!python train_bitsemcom.py --C {C} --init {INIT} --train_dir datasets/DIV2K_train_HR --val_dir datasets/kodak_patches256 --steps {STEPS} --batch {BATCH} --lr 1e-4 --min_lr 1e-6 --warmup 500 --snr_min {SNR_MIN} --snr_max {SNR_MAX} --val_every 1000 --patience 8 --amp_dtype {AMP} --num_workers 4 --max_hours {MAX_HOURS} --resume --out {OUT}

## 7. GRAND-in-the-loop training

Same recipe; between the bit mapper and the semantic decoder sits the (128,112) CRC code, the interleaver, the
channel and ORBGRAND with the training budget. The decoder sees GRAND's output (decoded blocks exact, abandoned
blocks soft fallback); the gradient to the encoder is the uncoded soft channel's Jacobian (`--grand_grad soft`),
the same gradient path as the benchmark run. The log line ends with the GRAND cost per step
(`| GRAND aband .. misc .. bler .. q .. 0.xxs`). If that time dominates, add `--grand_images 8` or lower
`--grand_budget`; if the process pool ever hangs, use `--grand_workers 0`.

In [ ]:
OUT = f"{RUNS}/C{C}_grand"
!python train_bitsemcom.py --C {C} --init {INIT} --train_dir datasets/DIV2K_train_HR --val_dir datasets/kodak_patches256 --steps {STEPS} --batch {BATCH} --lr 1e-4 --min_lr 1e-6 --warmup 500 --snr_min {SNR_MIN} --snr_max {SNR_MAX} --val_every 1000 --patience 8 --amp_dtype {AMP} --num_workers 4 --max_hours {MAX_HOURS} --resume --grand {CODE} --grand_budget {GRAND_BUDGET} --grand_fallback soft --grand_grad soft --grand_workers {NPROC} --out {OUT}

Optional third run with imperfect-CSI augmentation (receiver SNR error up to 3 dB, channel-estimate error
variance up to 0.1 per batch): copy the cell above, add `--snr_err_train 3 --h_err_train 0.1` (and
`--channel blockfading --fade_block 8` if fading is the target channel) and use `--out {RUNS}/C32_grand_csi`.

## 8. Training status and curves

In [ ]:
import csv, glob, os
runs = [r for r in sorted(glob.glob(f"{RUNS}/C{C}_*")) if os.path.exists(f"{r}/log.csv")]
for run in runs:
    rows = list(csv.DictReader(open(f"{run}/val.csv"))) if os.path.exists(f"{run}/val.csv") else []
    if rows:
        last = rows[-1]["step"]
        print(os.path.basename(run), "| last validation at step", last, "|",
              " ".join("%s@%s=%.2f" % (r["receiver"][:8], r["snr"], float(r["psnr"])) for r in rows if r["step"] == last))
RUN_ARGS = " ".join(runs)
!python experiments/plot_training.py {RUN_ARGS} --out {FIG}
display(IMG(f"{FIG}/training.png", width=1100))

## 9. Evaluation: models x conditions x SNR x receivers

Primary sweep at the query budget the GRAND model was trained with (its decoder learned the abandonment and
miscorrection statistics of that budget). Metrics: PSNR, MS-SSIM, LPIPS, CLIP similarity. Also saves example
reconstructions of the first 3 images at 4 dB to `results/images_C32/` on Drive.

In [ ]:
import os
CKPTS = f"--ckpt stage2={INIT}"
for name in ("bench", "grand", "grand_csi"):
    m = f"{RUNS}/C{C}_{name}/best.model"
    if os.path.exists(m):
        CKPTS += f" --ckpt {name}={m}"
print(CKPTS)
EVAL = f"{RESULTS}/eval_C{C}.jsonl"
!python experiments/evaluate.py --C {C} {CKPTS} --snrs {EVAL_SNRS} --conditions {EVAL_CONDITIONS} --paths noiseless,uncoded_soft,uncoded_hard,coded_soft --budget {GRAND_BUDGET} --save_images 3 --save_snr 4 --out {EVAL}
!python experiments/plot_results.py {EVAL} --out {FIG}
for name in ("quality_awgn", "quality_awgn_over3", "quality_fade8_h0.1", "gain_psnr", "grand_awgn"):
    display(IMG(f"{FIG}/{name}.png", width=1100))

## 10. Secondary sweep: large query budget

Fewer abandonments, many more miscorrections at low SNR than the models were trained with. Reported separately
(figures tagged `b1e5`).

In [ ]:
EVAL2 = f"{RESULTS}/eval_C{C}_b1e5.jsonl"
!python experiments/evaluate.py --C {C} {CKPTS} --snrs {EVAL_SNRS} --conditions awgn,fade8 --paths noiseless,uncoded_soft,coded_soft --budget 100000 --out {EVAL2}
!python experiments/plot_results.py {EVAL2} --out {FIG} --tag b1e5
display(IMG(f"{FIG}/gain_psnr_b1e5.png", width=900))

## 11. Reference decoder: bit-exact check of the Python ORBGRAND port against Duffy's MATLAB code

Octave runs Duffy's unmodified `bin_ORBGRAND1.m` / `landslide.m` on LLR blocks from real semantic bits through the
thesis channels (AWGN 3/5/7 dB, block fading 10 dB with and without channel-estimate error). ORBGRAND is
deterministic, so the expected result is zero mismatches in codewords, guess counts and abandonment flags.
Octave is slow (one pattern per loop iteration): expect 10 to 20 minutes.

In [ ]:
!apt-get -qq install -y octave > /dev/null
!bash matlab/fetch_grand_matlab.sh
GV = f"{RESULTS}/grand_verify"
!python experiments/grand_verify.py export --from_model --C {C} --out {GV}
!python experiments/grand_verify.py run --backend octave --out {GV}

## 12. Optional, slow: final numbers through the reference decoder

Every ORBGRAND call of this reduced sweep goes through Duffy's code in Octave (one Octave process per
transmission). Use it for the final table once the Python sweep above is settled.

In [ ]:
EVAL3 = f"{RESULTS}/eval_C{C}_octave.jsonl"
!python experiments/evaluate.py --C {C} {CKPTS} --snrs 4,6,8,10 --conditions awgn,fade8 --paths noiseless,uncoded_soft,coded_soft --budget {GRAND_BUDGET} --grand_backend octave --out {EVAL3}
!python experiments/plot_results.py {EVAL3} --out {FIG} --tag octave

## 13. Save results to GitHub

JSONL rows, figures and summary tables go into the repo under `results/` and are pushed to the branch. Checkpoints,
training logs and example images stay on Drive (they are gitignored). Afterwards `git pull` on your PC.

In [ ]:
import os
os.makedirs("results/figures", exist_ok=True)
!cp {RESULTS}/*.jsonl results/ 2>/dev/null || true
!cp -r {FIG}/. results/figures/
!git config user.name "{GIT_NAME}"; git config user.email "{GIT_EMAIL}"
!git add results/*.jsonl results/figures
!git commit -q -m "Add Colab results: C={C} benchmark vs GRAND-in-the-loop (plan v2)" || echo "nothing new to commit"
!git push -q https://{TOKEN}@github.com/{GITHUB_REPO} HEAD:{BRANCH}
!git log --oneline -1

## What to compare (the thesis questions, `docs/colab_runbook.md` section 6)

1. `bench` vs `stage2`: did the proper schedule improve the model (noiseless and uncoded curves)? Caveat: `stage2`
   was trained at -3..6 dB, `bench` at 0..10 dB.
2. `grand` vs `bench` with the `coded_soft` receiver: does training with GRAND in the loop change what the decoder
   makes of GRAND's output, especially at 0-6 dB?
3. `coded_soft` vs `uncoded_soft` under `awgn_over3` / `awgn_under3`: GRAND's decoded bits do not depend on the SNR
   estimate, the soft receiver's P(1) does (`gain_*.png`).
4. `fade8` vs `fade8_h0.1`: what channel-estimation error does to the block error rate and to image quality.
5. LPIPS and CLIP vs PSNR: does the ranking of receivers change with a perceptual or semantic metric?